[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jairomelo/aiOCR/blob/main/models/Qwen2.5-VL-7B/Qwen2.5-VL-7B_4bit.ipynb)

## Prerequisites

Runtime: Python 3, T4 GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# qwen2_5_vl requires transformers from source (not yet on PyPI stable)
%pip install -q "git+https://github.com/huggingface/transformers.git" accelerate bitsandbytes
%pip install -q "qwen-vl-utils[decord]==0.0.8"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 64.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 106.9 MB/s eta 0:00:0000:010:01


In [3]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

In [4]:
import torch

In [5]:
from pathlib import Path

WORKING_DIR = Path('/content/drive/MyDrive/aiOCR')
MODEL_NAME = 'Qwen/Qwen2.5-VL-7B-Instruct'

qc = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# max_pixels caps the token budget: 512*28*28 ≈ 400k px — enough detail for
# document OCR while staying within T4 VRAM alongside the 4-bit weights
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

processor = AutoProcessor.from_pretrained(
    MODEL_NAME, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=qc,
    device_map="auto",
    torch_dtype=torch.float16,
).eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

In [13]:
IMAGE_FILES = [
    WORKING_DIR / 'images/CCundinamarca/CCundinamarca_page_1.png',
    WORKING_DIR / 'images/CCundinamarca/CCundinamarca_page_46.png',
    WORKING_DIR / 'images/CO_18180627/CO_18180627_page_1.png',
    WORKING_DIR / 'images/dmcz_18250101/dmcz_18250101_page_1.png',
    WORKING_DIR / 'images/dmcz_18250101/dmcz_18250101_page_4.png',
    WORKING_DIR / 'images/el-redactor-1/el-redactor-1_page_1.png',
    WORKING_DIR / 'images/el-redactor-1/el-redactor-1_page_2.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_1.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_3.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_4.png',
    WORKING_DIR / 'images/AR_SR8V4R3/AR_SR8V4R3_4.jpg',
    WORKING_DIR / 'images/fpineda_30_pza2/fpineda_30_pza2_page_2.png',
    WORKING_DIR / 'images/fpineda_184_pza6/fpineda_184_pza6_page_1.png',
    WORKING_DIR / 'images/fpineda_196_pza8/fpineda_196_pza8_page_12.png'
]

## Inference

In [14]:
import time

transcription_out = WORKING_DIR / 'transcriptions/Qwen2.5-VL-7B'
transcription_out.mkdir(parents=True, exist_ok=True)

prompt = (
    "Convert the document to plain text, as close to the original as possible "
    "(including typos, print errors, and original grammar and spelling). "
    "Do not add any formatting, markdown, or annotations."
)

for IMAGE_FILE in IMAGE_FILES:
    image_stem = IMAGE_FILE.stem
    out_path = transcription_out / f'{image_stem}.md'

    if out_path.exists():
        print(f'Skipping (already done): {image_stem}')
        continue

    if not IMAGE_FILE.exists():
        print(f'Skipping (image not found): {IMAGE_FILE}')
        continue

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": str(IMAGE_FILE)},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    t0 = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=4096,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
            do_sample=True,
        )
    elapsed = time.time() - t0

    generated_ids_trimmed = [out[input_len:] for out in generated_ids]
    transcription = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    out_path.write_text(transcription, encoding='utf-8')
    print(f'Done in {elapsed:.1f}s — saved: transcriptions/Qwen2.5-VL-7B/{image_stem}.md')

Done in 25.7s
¡A DONDE VAMOS A PARAR?

A la familia i à cada uno de sus miembros, à los padres,
à los hijos, à los jóvenes, à los ancianos,

¿Qué daño os ha hecho?

I.

ACERCABASE la hora fatal: las potestades de las tinieblas
se habian desenfrenado; i he aqui que todo un pueblo
dominado de un espíritu de furor i de vértigo se apodera
del Justo. Los propios discípulos de este, educados en
su escuela, alimentados con su pan, colmados de caricias,
sus discípulos que acaban de jurarle una fidelidad á toda
prueba, le abandonan i le niegan: uno de ellos le ha
vendido. Atado como un malhechor es conducido de
tribunal en tribunal por las calles de una gran ciudad.
Hombres, mujeres, niños, magistrados, ancianos con los
cabellos blancos, todos han acudido i forman la tumultuaria
comitiva. De entre aquella multitud horrible i como un
hombre ébrio i ajustada como un mar borrascoso salen
inscansablemente gritos de muerte. El odio impaciente no


### Saving the output